# 04. Build a validation sample (full opinion text)

Pulls a random sample of **control-eligible** opinions from the bulk file, keeping enough opinion
text around the sanctions ruling that a human can actually judge the severity. This is the input
for hand-validating the control severity coder — the earlier `controls_coded.csv` only kept a
300-char excerpt, which is too thin to code from.

**Outputs** (to `../data/coded/`):
- `controls_validation_200.csv` — 200 random cases with a wide text window + the coder's label + a
  blank `hand_severity` column for you to fill.
- `controls_validation_40.csv` — a 40-row random subset (a quick first pass).

Run from `code/`. Needs the bulk file at the same path `02` uses.

### Config

In [1]:
import re, sys, os
import pandas as pd, numpy as np
sys.path.append("labeling"); import label_lib as L

SNAPSHOT_DATE = "2026-06-30"
BULK_CSV = f"../data/raw/bulk-data/opinion-clusters-{SNAPSHOT_DATE}.csv.bz2"
YEAR_MIN, YEAR_MAX = 2023, 2026
N_SAMPLE, N_QUICK, SEED = 200, 40, 11
CHUNKSIZE = 100_000

SANCTION_TRIGGER = re.compile(
    r"rule\s*11|§?\s*1927|section\s*1927|inherent\s+authority|sanction|"
    r"show\s+cause|disciplin|referr", re.I)
READ_KW = dict(engine="c", quotechar='"', escapechar="\\", on_bad_lines="skip", low_memory=False, dtype=str)
COL = dict(case="case_name", date="date_filed", nos="nature_of_suit", attorneys="attorneys",
           text_cols=["disposition","summary","procedural_history","posture","syllabus","headnotes"])
KEEP = set([COL["case"],COL["date"],COL["nos"],COL["attorneys"]]+COL["text_cols"])

### Helpers

In [2]:
def build_text(df):
    cols=[c for c in COL["text_cols"] if c in df.columns]
    s=pd.Series("", index=df.index, dtype="object")
    for c in cols:
        s=s.str.cat(df[c].fillna("").astype(str), sep="  ")
    return s

def wide_region(text, before=400, after=2200):
    """A generous window around the sanctions trigger, so a human can read the ruling."""
    if not isinstance(text,str) or not text: return ""
    m=SANCTION_TRIGGER.search(text)
    if not m: return text[:1500]
    i=m.start(); return text[max(0,i-before): i+after]

def eligible(df):
    """Same population as the controls: window + sanctions trigger, minus AI-contaminated and
    bar-discipline. Returns rows with a full text blob to sample from."""
    df=df.rename(columns={COL["case"]:"case_name", COL["date"]:"date_filed", COL["nos"]:"nature_of_suit"})
    df["date"]=pd.to_datetime(df.get("date_filed"), errors="coerce"); df["year"]=df["date"].dt.year
    df=df[df["year"].between(YEAR_MIN,YEAR_MAX)]
    if len(df)==0: return pd.DataFrame(columns=["case_name","date_filed","year","nature_of_suit","_text"])
    df["_text"]=build_text(df)
    keep=(df["_text"].str.contains(SANCTION_TRIGGER,na=False)
          & ~df["_text"].map(L.is_ai_contaminated)
          & ~df["_text"].map(L.is_bar_discipline))
    df=df[keep]
    return df[["case_name","date_filed","year","nature_of_suit","_text"]]

### Stream the bulk file and collect eligible cases

In [3]:
assert os.path.exists(BULK_CSV), f"Not found: {BULK_CSV}"
reader=pd.read_csv(BULK_CSV, usecols=lambda c: c in KEEP, chunksize=CHUNKSIZE, **READ_KW)
pool=[]; scanned=0
for i,ch in enumerate(reader):
    pool.append(eligible(ch)); scanned+=len(ch)
    if (i+1)%10==0: print(f"  scanned {scanned:>10,} | eligible so far {sum(len(p) for p in pool):>5,}")
elig=pd.concat(pool, ignore_index=True) if pool else pd.DataFrame()
print(f"\ntotal eligible controls: {len(elig):,}")

  scanned  1,000,000 | eligible so far   256
  scanned  2,000,000 | eligible so far   448
  scanned  3,000,000 | eligible so far   575
  scanned  4,000,000 | eligible so far   621
  scanned  5,000,000 | eligible so far   643
  scanned  6,000,000 | eligible so far   668
  scanned  7,000,000 | eligible so far   700
  scanned  8,000,000 | eligible so far   825
  scanned  9,000,000 | eligible so far   891
  scanned 10,000,000 | eligible so far   948

total eligible controls: 952


### Sample, label with the coder, write the validation files

In [4]:
samp=elig.sample(min(N_SAMPLE,len(elig)), random_state=SEED).reset_index(drop=True)
samp["coder_severity"]=samp["_text"].map(lambda t: L.code_severity(t, is_full_opinion=True))
samp["text_for_coding"]=samp["_text"].map(wide_region)
samp["hand_severity"]=""     # <- you (or Claude) fill this reading text_for_coding
out_cols=["case_name","date_filed","year","nature_of_suit","coder_severity","hand_severity","text_for_coding"]

samp[out_cols].to_csv("../data/coded/controls_validation_200.csv", index=False)
samp[out_cols].sample(min(N_QUICK,len(samp)), random_state=SEED).to_csv(
    "../data/coded/controls_validation_40.csv", index=False)
print("wrote ../data/coded/controls_validation_200.csv and controls_validation_40.csv")
print("coder_severity dist in sample:", samp["coder_severity"].value_counts().sort_index().to_dict())

wrote ../data/coded/controls_validation_200.csv and controls_validation_40.csv
coder_severity dist in sample: {0: 164, 1: 1, 2: 9, 3: 12, 4: 14}
